# BQP-GUB: the paper's methods on your dataset (Jupyter)

Runs **AADMM, PAL, MADMM-ALU** (all with local-search polishing) and the baselines **FISTA** (relaxation without ADMM),
**multistart LS**, **greedy+LS** and **ILS** on every instance, and writes one CSV row per instance.
Put `bqpgub_paper_v4.py` in the same folder as this notebook. The long job runs in a **background process**:
you may close the notebook and re-run the *Monitor* cell later; *Resume* continues where it stopped.

## 1. Configuration (edit this cell)

In [ ]:
from pathlib import Path

DATA_DIR   = Path("data")           # folder with your instance files
PATTERN    = "*"                    # e.g. "*.npz"
OUT        = "paper_results.csv"    # results file (appended to; safe to resume)
LOG        = "paper_run.log"

METHODS    = "aadmm,pal,madmm,fista,ls_ms,greedy,ils"   # add ",pal_fixed" for the PAL ablation (3 extra PAL runs)
TIME_LIMIT = 300      # wall-clock cap per method, seconds (hits are flagged in the CSV)
MATCH_CAP  = 120      # cap (s) on the matched time budget of ls_ms and ils (0 = match the slowest ADMM-type method)
THREADS    = "1"      # BLAS threads; "1" = single-thread (as in the paper). Larger = faster for n >= 2000
MAX_N      = 0        # skip instances with n above this (0 = none): use e.g. 2000 for a first pass
SEED       = 0

## 2. Environment check

In [ ]:
import sys, os, importlib, re, subprocess, signal, time
print("python:", sys.executable, "| CPU cores:", os.cpu_count())
for mod in ["numpy", "scipy", "pandas"]:
    try: m = importlib.import_module(mod); print(f"{mod:7s} OK {m.__version__}")
    except Exception: print(f"{mod:7s} MISSING -> %pip install {mod}")
print("bqpgub_paper_v4.py", "found" if Path("bqpgub_paper_v4.py").exists() else "NOT FOUND (copy it next to this notebook)")
def natural(p): return [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", p.name)]
files = sorted((p for p in DATA_DIR.glob(PATTERN) if p.is_file()), key=natural)
print(len(files), "instance files in", DATA_DIR.resolve())

## 3. Pilot on three files (about a minute)
Checks that your files are read and that all methods run. If you see `[skip] ...`, send me the first 10-15 lines of that file.

In [ ]:
pilot = [files[0], files[len(files)//2], files[-1]] if len(files) >= 3 else files
print("pilot:", [p.name for p in pilot])
if Path("pilot.csv").exists(): Path("pilot.csv").unlink()
cmd = [sys.executable, "bqpgub_paper_v4.py", *map(str, pilot), "--methods", METHODS, "--time-limit", "30",
       "--match-cap", "10", "--threads", THREADS, "--seed", str(SEED), "--out", "pilot.csv"]
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout[-3500:]); print(res.stderr[-1500:])

## 4. Launch the full run in the background (safe to re-run: `--resume`)

In [ ]:
cmd = [sys.executable, "bqpgub_paper_v4.py", str(DATA_DIR), "--pattern", PATTERN, "--methods", METHODS,
       "--time-limit", str(TIME_LIMIT), "--match-cap", str(MATCH_CAP), "--threads", THREADS,
       "--max-n", str(MAX_N), "--seed", str(SEED), "--out", OUT, "--resume"]
proc = subprocess.Popen(cmd, stdout=open(LOG, "a"), stderr=subprocess.STDOUT, start_new_session=True)
Path("paper_run.pid").write_text(str(proc.pid)); print("started, PID", proc.pid, "| log:", LOG, "| results:", OUT)

## 5. Monitor (re-run any time)

In [ ]:
import pandas as pd
def alive(pid):
    try: os.kill(pid, 0); return True
    except OSError: return False
pid = int(Path("paper_run.pid").read_text()) if Path("paper_run.pid").exists() else None
print("background job running:", alive(pid) if pid else "no job started")
if Path(OUT).exists():
    df = pd.read_csv(OUT); print(f"done: {len(df)} / {len(files)} instances")
    hits = [c for c in df.columns if c.endswith("_hit")]
    if hits:
        print("time-limit hits per method:", {c[:-4]: int(df[c].astype(str).str.upper().eq("TRUE").sum()) for c in hits})
    show = [c for c in ["name","n","aadmm_fls","pal_fls","madmm_fls","lsms_f","ils_f","aadmm_t","pal_t","madmm_t"] if c in df]
    display(df[show].tail(8).round(2))
print("\n--- log tail ---"); print("".join(open(LOG).readlines()[-6:]) if Path(LOG).exists() else "(no log yet)")

## 6. Stop the background job (optional)

In [ ]:
if Path("paper_run.pid").exists():
    try: os.killpg(os.getpgid(int(Path("paper_run.pid").read_text())), signal.SIGTERM); print("stopped")
    except Exception as e: print("not running:", e)

## 7. Summary table (works on partial results; writes `paper_tables.tex`)

In [ ]:
res = subprocess.run([sys.executable, "bqpgub_paper_v4.py", "--summarize", OUT, "--latex", "paper_tables.tex"],
                     capture_output=True, text=True)
print(res.stdout); print(res.stderr[-1500:])

## 8. Send me
`paper_results.csv` (and any `[skip]` lines). I will update the tables, the text, the abstract and the conclusion to match what the data show.